# Mongoose Schemas and Models

A **schema** defines the shape, data types and validation rules of your documents. A **model** is the compiled, callable class you actually query with. Schema describes; model does.

```
new mongoose.Schema({...})  →  mongoose.model('User', schema)  →  User.find(), new User(), ...
```

## Basic implementation

```javascript
const mongoose = require('mongoose');

// 1. Define the schema structure
const userSchema = new mongoose.Schema({
  username: {
    type: String,
    required: true,
    unique: true,
    trim: true
  },
  email: {
    type: String,
    required: true
  },
  age: {
    type: Number,
    min: 18
  },
  role: {
    type: String,
    enum: ['user', 'admin'],
    default: 'user'
  }
}, {
  timestamps: true // Automatically adds createdAt and updatedAt fields
});

// 2. Compile the schema into a model
const User = mongoose.model('User', userSchema);

module.exports = User;
```

Shorthand — when you need no options beyond the type, the config object is optional:

```javascript
const userSchema = new mongoose.Schema({
  username: String,          // same as { type: String }
  tags: [String],            // array of strings
  meta: { visits: Number }   // nested object
});
```

## SchemaTypes

`String`, `Number`, `Boolean`, `Date`, `Buffer`, `Array`, `Map`, `Decimal128`, `BigInt`, `Schema.Types.ObjectId`, `Schema.Types.Mixed`.

Each type brings its own options:

| Type | Options |
|---|---|
| String | `lowercase`, `uppercase`, `trim`, `match` (regex), `enum`, `minLength`, `maxLength` |
| Number | `min`, `max`, `enum` |
| Date | `min`, `max`, `expires` (TTL index) |
| ObjectId | `ref` / `refPath` — the model to populate from |
| Mixed | No schema enforcement; must call `doc.markModified(path)` before saving |

Options available on every type: `required`, `default`, `select`, `validate`, `immutable`, `index`, `unique`, `sparse`, `alias`, `transform`.

`default` accepts a function, which is how you get a fresh value per document:

```javascript
createdAt: { type: Date, default: Date.now }  // note: no parentheses — pass the fn, don't call it
```

## Validation

### Built-in validators

Pass a config object instead of a raw type to enforce `required`, `min`/`max`, `enum`, `minLength`/`maxLength`, `match`.

Custom messages use array or object form:

```javascript
age: {
  type: Number,
  min: [18, 'Must be at least 18, got {VALUE}'],
  max: 120
},
role: {
  type: String,
  enum: {
    values: ['user', 'admin'],
    message: '{VALUE} is not a valid role'
  }
}
```

> **`unique` is not a validator.** It's shorthand for building a unique *index* in MongoDB. It doesn't run in Mongoose's validation pipeline, so a duplicate throws a `MongoServerError` with code `11000` from the driver, not a `ValidationError`. It also only takes effect once the index exists — with `autoIndex: false` in production you must build it yourself.

### Custom validators

```javascript
email: {
  type: String,
  required: true,
  validate: {
    validator: v => /^\S+@\S+\.\S+$/.test(v),
    message: props => `${props.value} is not a valid email`
  }
}
```

Async validators work too — return a promise from `validator`.

### When validation runs

Validators run on `save()` and on `create()`. They **do not** run on `findOneAndUpdate()`, `updateOne()` or `updateMany()` unless you opt in:

```javascript
await User.findByIdAndUpdate(id, { age: 5 }, { runValidators: true, new: true });
```

Even then, `this` isn't a document in an update validator, so custom validators that read sibling fields behave differently. This trips up almost everyone once.

## Schema options (second argument)

```javascript
new mongoose.Schema({ /* ... */ }, {
  timestamps: true,       // adds createdAt / updatedAt
  strict: true,           // default: silently drop fields not in the schema
  versionKey: '__v',      // set false to remove the version key
  collection: 'app_users',// override the derived collection name
  toJSON: { virtuals: true },
  toObject: { virtuals: true },
  id: true                // virtual `id` getter returning _id as a string
});
```

`timestamps` can be renamed: `{ timestamps: { createdAt: 'created_at', updatedAt: false } }`.

## Automatic `_id`

Mongoose assigns a unique `ObjectId` to `_id` on every new document. An ObjectId embeds its creation time, so `doc._id.getTimestamp()` gives you a `Date` for free.

Disable on subdocuments with `{ _id: false }` when you don't need per-item ids.

Comparing ids: `Types.ObjectId` is an object, so `id1 === id2` is false even for equal ids. Use `id1.equals(id2)` or compare `.toString()`.

## Collection naming

`mongoose.model('User', userSchema)` makes Mongoose look for the lowercase, pluralised collection — `users`. The pluraliser handles irregulars reasonably (`Person` → `people`, `Category` → `categories`) but not always the way you'd expect.

Override explicitly when it matters:

```javascript
mongoose.model('User', userSchema, 'app_users');       // third argument
// or via schema options: { collection: 'app_users' }
```

The model *name* is also what `ref` strings point at, so keep it singular and PascalCase.

## Nested structures

```javascript
const addressSchema = new mongoose.Schema({
  street: String,
  city: { type: String, required: true }
}, { _id: false });

const orderSchema = new mongoose.Schema({
  shipping: addressSchema,                    // single subdocument
  items: [{                                   // array of subdocuments
    product: { type: Schema.Types.ObjectId, ref: 'Product' },
    qty: { type: Number, default: 1, min: 1 }
  }],
  tags: [String],                             // array of primitives
  scores: { type: Map, of: Number }           // dynamic keys
});
```

Arrays default to `[]`, not `undefined` — so `required` on an array passes even when it's empty. Validate length explicitly if you need at least one element.

## Adding behaviour to a schema

All of these must be declared **before** `mongoose.model()` is called.

```javascript
// Instance method — available on documents
userSchema.methods.getInitials = function () {
  return this.username.slice(0, 2).toUpperCase();
};

// Static — available on the model
userSchema.statics.findByEmail = function (email) {
  return this.findOne({ email });
};

// Virtual — computed, not persisted
userSchema.virtual('isAdult').get(function () {
  return this.age >= 18;
});

// Query helper — chainable
userSchema.query.admins = function () {
  return this.where({ role: 'admin' });
};
```

Use `function` and not arrow functions here — Mongoose binds `this` to the document/model/query.

Virtuals are excluded from `JSON.stringify()` output unless you set `toJSON: { virtuals: true }` in the schema options.

## Middleware (hooks)

```javascript
userSchema.pre('save', async function (next) {
  if (!this.isModified('password')) return next();
  this.password = await bcrypt.hash(this.password, 12);
  next();
});

userSchema.post('findOneAndDelete', async function (doc) {
  if (doc) await Order.deleteMany({ user: doc._id });
});
```

Document middleware (`save`, `validate`) has `this` as the document. Query middleware (`find`, `findOneAndUpdate`, `deleteOne`) has `this` as the *query* — the document isn't loaded, which is why `pre('save')` password hashing silently doesn't fire on `findOneAndUpdate`.

## Indexes

```javascript
userSchema.index({ email: 1 }, { unique: true });
userSchema.index({ createdAt: -1 });
userSchema.index({ username: 'text', bio: 'text' });   // text search
```

Mongoose builds these automatically at startup via `autoIndex`, which is convenient in development and a liability in production on large collections — disable it there and manage index builds deliberately.

## TypeScript implementation

Declare an interface for the document properties and pass it as a generic to `Schema` and `model`:

```typescript
import { Schema, model, Types } from 'mongoose';

// 1. Define the interface
interface IUser {
  name: string;
  email: string;
  organization: Types.ObjectId; // For references
  avatar?: string;
}

// 2. Create the schema matching the interface
const userSchema = new Schema<IUser>({
  name: { type: String, required: true },
  email: { type: String, required: true },
  organization: { type: Schema.Types.ObjectId, ref: 'Organization' },
  avatar: String
});

// 3. Compile the model
export const User = model<IUser>('User', userSchema);
```

### Inferring the type instead

If you'd rather not maintain the interface by hand, let Mongoose derive it:

```typescript
import { Schema, model, InferSchemaType } from 'mongoose';

const userSchema = new Schema({
  name: { type: String, required: true },
  age: Number
});

type IUser = InferSchemaType<typeof userSchema>;
export const User = model('User', userSchema);
```

Hand-written interfaces give better control over optionality; inference guarantees the type never drifts from the schema. Pick one and be consistent.

### Typing methods and statics

```typescript
interface IUserMethods {
  getInitials(): string;
}
interface UserModel extends Model<IUser, {}, IUserMethods> {
  findByEmail(email: string): Promise<HydratedDocument<IUser, IUserMethods> | null>;
}

const User = model<IUser, UserModel>('User', userSchema);
```

`HydratedDocument<IUser>` is the type of an actual document instance — `IUser` plus `_id`, `save()`, and the rest of the document API.

## Gotchas

| Symptom | Cause |
|---|---|
| Field silently missing after save | Not declared in the schema; `strict: true` drops it |
| `E11000 duplicate key error` | Unique index violation — catch code `11000`, it's not a `ValidationError` |
| Validators don't fire on update | Need `{ runValidators: true }` on `findOneAndUpdate` etc. |
| `OverwriteModelError` | `mongoose.model()` called twice with the same name (usually hot reload) |
| `this` is undefined in a method | Arrow function used where Mongoose needs `function` |
| Virtual missing from API response | `toJSON: { virtuals: true }` not set |
| Mixed-type changes not persisting | Call `doc.markModified('path')` |
| Every `_id` comparison is false | Comparing ObjectIds with `===` instead of `.equals()` |

## Sources

- [Schemas guide](https://mongoosejs.com/docs/guide.html)
- [Models](https://mongoosejs.com/docs/models.html)
- [SchemaTypes](https://mongoosejs.com/docs/schematypes.html)
- [Validation](https://mongoosejs.com/docs/validation.html)
- [TypeScript support](https://mongoosejs.com/docs/typescript.html)